# [Day 17] Advanced Training: FastAPI 실전 CRUD (총 3문제)

**"가상의 데이터베이스(리스트+딕셔너리)를 조작하는 진짜 API 만들기"**

Basic에서 다진 딕셔너리 컨트롤과 예외 처리 감각을 FastAPI 라우터에 적용해 봅니다.

---

### Q1. 특정 유저 찾기 (GET & 예외 처리)
유저 ID를 Path Variable로 입력받아 해당 유저를 반환하는 `get_user` API를 만드세요.
만약 해당 ID의 유저가 없다면, `raise HTTPException(status_code=404, detail="User not found")`를 발생시켜야 합니다.

In [1]:
from fastapi import FastAPI, HTTPException

app = FastAPI()

# 가상의 데이터베이스 (리스트 안에 딕셔너리)
fake_db = [
    {"id": 1, "username": "Alice", "age": 25},
    {"id": 2, "username": "Bob", "age": 30}
]

@app.get("/users/{user_id}")
def get_user(user_id: int):
    for data in fake_db:
        if user_id == data['id']:
            return data

    raise HTTPException(status_code=404, detail="User not found")


# 테스트 (주피터 환경이므로 함수를 직접 호출해 봅니다)
print("Q1 존재하는 유저 찾기:", get_user(1))
try:
    get_user(999)
except HTTPException as e:
    print(f"Q1 에러 캐치 성공: {e.status_code} - {e.detail}")

Q1 존재하는 유저 찾기: {'id': 1, 'username': 'Alice', 'age': 25}
Q1 에러 캐치 성공: 404 - User not found


### Q2. 유저 삭제하기 (DELETE)
유저 ID를 입력받아 `fake_db`에서 해당 유저를 삭제하는 API를 만듭니다.
이번에는 파이썬의 강력한 문법인 **리스트 컴프리헨션(List Comprehension)**을 사용하여, 삭제할 ID가 *아닌* 유저들만 모아서 `fake_db`를 갱신해 보겠습니다.

In [2]:
@app.delete("/users/{user_id}")
def delete_user(user_id: int):
    global fake_db
    
    # 삭제 전 DB 길이 저장
    original_length = len(fake_db)
    
    fake_db = [data for data in fake_db if data['id'] != user_id]

    
    if len(fake_db) == original_length:
        raise HTTPException(status_code=404, detail="User not found")
        
    return {"message": "User deleted successfully"}

# 테스트
print("Q2 삭제 성공 여부:", delete_user(1))
print("Q2 삭제 후 DB 상태:", fake_db)

Q2 삭제 성공 여부: {'message': 'User deleted successfully'}
Q2 삭제 후 DB 상태: [{'id': 2, 'username': 'Bob', 'age': 30}]


### Q3. 유저 정보 수정하기 (PUT)
Pydantic 모델을 사용하여 클라이언트로부터 수정할 나이(`age`)를 전달받고, 해당 유저의 나이를 업데이트하는 API를 만듭니다.

In [6]:
from pydantic import BaseModel

class UserUpdate(BaseModel):
    age: int

@app.put("/users/{user_id}")
def update_user_age(user_id: int, user_data: UserUpdate):
    for data in fake_db:
        if data['id'] == user_id:
            data['age'] = user_data.age
            return data
    raise HTTPException("에러")
# 테스트
update_payload = UserUpdate(age=99)
print("Q3 업데이트 성공:", update_user_age(2, update_payload))
print("Q3 업데이트 후 DB 상태:", fake_db)

Q3 업데이트 성공: {'id': 2, 'username': 'Bob', 'age': 99}
Q3 업데이트 후 DB 상태: [{'id': 2, 'username': 'Bob', 'age': 99}]
